**Load Dataset**

In [ ]:
from google.colab import userdata
import os

os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_TOKEN')

In [ ]:
os.makedirs('/root/.kaggle', exist_ok=True)

token = 'KGAT_xxxxxxxxxxxxxxxxxxxx'
with open('/root/.kaggle/access_token', 'w') as f:
    f.write(token)

os.chmod('/root/.kaggle/access_token', 0o600)

!pip install kaggle -q
!kaggle datasets download -d kazanova/sentiment140

import zipfile
with zipfile.ZipFile('sentiment140.zip', 'r') as zip_ref:
    zip_ref.extractall('.')

print("✅ Done!")

**Install & Import Libraries**

In [ ]:
import numpy as np
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import swifter
import nltk
nltk.download('stopwords')

In [ ]:
twitt_data = pd.read_csv(
    "/content/training.1600000.processed.noemoticon.csv",
    encoding="latin-1",
    header=None,
    usecols=[0, 5]
)
twitt_data.columns = ["target", "text"]

**Clean Dataset**

In [ ]:
print(twitt_data.shape)
twitt_data.head()

In [ ]:
twitt_data[['target','text']].isnull().sum()

In [ ]:
twitt_data['target'].value_counts()

In [ ]:
twitt_data["target"] = twitt_data["target"].replace(4, 1)
twitt_data['target'].value_counts()

**Text Preprocessing**

In [ ]:
stemmer = PorterStemmer()
stop_words = set(stopwords.words("english"))

def stemming(content):
    try:
        stemmed_content = re.sub('[^a-zA-Z]', ' ', content).lower()
        return ' '.join(
            stemmer.stem(word)
            for word in stemmed_content.split()
            if word not in stop_words
        )
    except:
        return ""

In [ ]:
twitt_data['stem_text'] = twitt_data['text'].swifter.apply(stemming)

**Train/Test Split**

In [ ]:
X = twitt_data['text']
y = twitt_data['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)

**TF-IDF Vectorization**

In [ ]:
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

**Train Logistic Regression**

In [ ]:
model = LogisticRegression(
    max_iter=1000,
    solver='saga',
    n_jobs=-1
)

model.fit(X_train_tfidf, y_train)

In [ ]:
y_pred = model.predict(X_test_tfidf)
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

**Test With Custom Tweets**

In [ ]:
def predict_sentiment(tweet):
    cleaned = stemming(tweet)
    vectorized = tfidf.transform([cleaned])
    prediction = model.predict(vectorized)[0]
    confidence = model.predict_proba(vectorized)[0]

    print(tweet)
    print("Positive" if prediction == 1 else "Negative")
    print(max(confidence))